In [ ]:
#%% GENERAL LIBRARIES

# General
import sys
import os
from os.path import dirname, abspath
from glob import glob
import numpy as np
import pandas as pd
from osgeo import gdal, osr

# Plot
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties
import matplotlib as mpl
from matplotlib.dates import YearLocator, MonthLocator, DateFormatter
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import LightSource
from matplotlib.pyplot import cm
from matplotlib.ticker import MaxNLocator

# Gis
from osgeo import gdal
import imageio
import rasterio
import geopandas as gpd
import whitebox
wbt = whitebox.WhiteboxTools()
wbt.verbose = True

# Warnings
import logging
import warnings
logging.captureWarnings(True)


In [ ]:
#%% HYDROMODPY MODULES

# Class
from watershed import watershed_root, forcing, watershed_display
from tools import toolbox
from watershed.data import hydrology, climatic, oceanic, piezometry
from groundwater_flow import modflow_display

# Plot
fontprop = toolbox.plot_params(8,15,18,20) # small, medium, interm, large


In [ ]:
#%% NECESSARY INPUT PATHS

# User
user = 'Ronan'

# Path where the results will be stored (SHOULD BE SPECIFIED BY THE USER)
if user == 'Jean-Raynald':
    out_path = "D:/results/HydroModPy/"
if user == 'Alexandre':
    git_path = "C:/Users/alexa/Documents/GitHub/HydroModPy/CORE_COMM/"
    out_path = "C:/Users/alexa/Dropbox/HydroModPy/"
if user == 'Martin':
    out_path = 'C:/Users/Martin/Desktop/Travail/HydroModPy/output2/'
if user == 'Ronan':
    git_path = "D:/Users/abherve/GITHUB/HydroModPy/CORE_COMM/"
    out_path = 'D:/Users/abherve/TEST/'

# Git path
sys.path.append(git_path)

# Path to the test folder
test_path = git_path + "/tests/1_given data/"

# We suggest that data be stored in the following suite of specific folders
# 1 folder for each of the type of data and "process" to be simulated
dems_path = test_path + 'dem/'
hydrology_path = test_path + 'hydrology/'   # add hydrographic shapefiles
modflow_path = test_path + 'modflow/'       # add bin/ folder with necessary .exe
climate_path =test_path + 'climate/'
intermittency_path = test_path + 'intermittency/'
hydrometry_path = test_path + 'hydrometry/'
piezometry_path = None                      # add piezometry data or nothing for automatic download
geology_path = None                         # add geologic layers
oceanic_path = 'None'                         # add specific sea level files

# Specifically designed to process SURFEX data (France scale)
surfex_path =  None # add surfex models in .h5 format

# Indicate the name of the regional DEM
dem_name = "DEM_test_75m_LAMB93.tif"
dem_path = dems_path + dem_name

dem = gdal.Open(dem_path)
proj = osr.SpatialReference(wkt=dem.GetProjection())    # Retrieves projection system attached to the dem
crs = int(proj.GetAttrValue('AUTHORITY',1))             # Gets name of the projection system

# Import the library of watersheds (maybe several watersheds in the loaded file: library of watersheds)
library_path = test_path + 'watershed_library.csv' # each row is a study site
library = pd.read_csv(library_path, sep=';', header=0, engine='python') # explore catchment studied

# Selection of the watershed to deal within from the just loaded library of watersheds
watershed_name = 'Example' # add manually study site information in map units  #JR:Parameters
#RONAN: Supprimer la ligne?
mysite = library[library['watershed_name'] == watershed_name] # specific row

# Paths generated automatically but necessary for plots
stable_folder = out_path+'/'+watershed_name+'/'+'results_stable/'
simulations_folder = out_path+'/'+watershed_name+'/'+'results_simulations/'


In [ ]:
#%% GENERATING WATERSHED

load = False
print('##### '+watershed_name.upper()+' #####')

subbasin_path = True   # generate subbasins from stations or manual points
from_shp = None        # specify a path if process start from a given shapefile
from_dem = False       # True or False if the process start from a given DEM of xyz file
cell_size = None       # specify new resolution from a given DEM or None
from_xy = []

try:
    BV = watershed_root.Watershed(watershed_name=watershed_name,
                                  dem_path=dem_path, 
                                  out_path=out_path,
                                  modflow_path=modflow_path,
                                  library_path=library_path,
                                  load=load,
                                  from_shp=from_shp,
                                  from_dem=from_dem,
                                  from_xy=from_xy,
                                  cell_size=cell_size)
except:
    print('There is a problem to generate the watershed object')


In [ ]:
#%% ADD SPECIFIC DATA

# Specify the hydrologic layers to clip
types_obs = ['streams','sections'] # list of shapefile name layers  #JR:Parameters
fields_obs = ['FID','Persistanc'] # list of shapefile name columns to translate in a tif #JR:Parameters

BV.add_hydrology(hydrology_path, types_obs=types_obs, fields_obs=fields_obs)

BV.add_hydrodynamic()
BV.add_forcing()
BV.add_oceanic(oceanic_path)

watershed_display.watershed_dem(BV)
watershed_display.watershed_local(dem_path, BV)

In [ ]:
#%% SET PARAMETERS

# Choice the state of the simulation
sim_state = 'steady' # steady
first = 2010
last = 2019
time_step = 'M'

# Recharge from a csv
rec = pd.read_csv(climate_path+'_REC_'+time_step+'.csv', sep=';', index_col=[0], parse_dates=True)
rec = rec[(rec.index.year>=first) & (rec.index.year<=last)]
rec = rec.squeeze()
BV.forcing.update_recharge(values = rec / 1000, sim_state=sim_state)

# Finally the rehcarge is set as a value or a serie
R = BV.forcing.recharge # mm/month to m/month

# Plot to control recharge
if sim_state == 'transient':
    fig, ax = plt.subplots(1,1, figsize=(8,3))
    ax.plot(R*1000, c='k', lw=0.5)

# Update hydrualic conductivity
K = 1e-5 * 3600 * 24 * 30 # m/second to m/month
BV.hydrodynamic.update_hyd_cond(K)

# Update aquifer thickness
E = 30 # m
BV.hydrodynamic.update_thickness(E)

# Update effective porosity
P = 0.01 # -
BV.hydrodynamic.update_porosity(P)

# Set name of the model
model_name = sim_state

In [ ]:
#%% RUN MODEL

# Launch a model
success, flow_model= BV.run_modflow(ident=model_name, modpath_sim=True, first_only=True,
                                    sink_fill=False, box=False,
                                    lay_number=1, bottom=None, thick_exp=1., cond_decay=0., 
                                    verbose=True)
print('Modeling process completed')


In [ ]:
#%% POST-PROCESSING

BV.matrix_modflow(success,
                  flow_model,
                  first_only = True,
                  watertable_elevation = True,
                  watertable_depth = True, 
                  seepage_areas = True,
                  outflow_drain = True,
                  groundwater_flux = False,
                  specific_discharge = False,
                  accumulation_flux = True,
                  perenn_intermit_shp = False,
                  groundwater_storage = True,
                  residence_times = True,
                  verbose = True,
                  export_tif = True)

# Extract result chronics
BV.results_modflow(ident=model_name, actual_date=True, time_step='M')
print('Result chronics extraction completed')


In [ ]:
#%% PLOT SURFACE OUTPUTS

if sim_state == 'transient':
    modflow_display.SurfaceOutputs(R, simulations_folder, stable_folder, model_name,
                                   types_obs, freq_interv=12, save_gif=True)
if sim_state == 'steady':
    # Control plot
    x = np.load(simulations_folder+'/test/_watershed/accumulation_flux.npy', allow_pickle=True).item()
    x = x[0]
    x[x<=0] = np.nan
    plt.imshow(x, cmap='jet')
    

In [ ]:
#%% INTERACTIVE CROSS-SECTION

# Dem data
dem_data = BV.geographic.dem_data
# dem_data = imageio.imread(stable_folder+'/geographic/'+'watershed_box_buff_dem.tif')
# dem_data = imageio.imread(stable_folder+'/geographic/'+'watershed_dem.tif')

# Wt data
wt_data = imageio.imread(simulations_folder+model_name+'/_watershed/_tifs/'+'watertable_elevation_t(0).tif') # buffer size no masked

# River data
river_data = imageio.imread(stable_folder+'/hydrology/'+'sections.tif')

# Function
modflow_display.interactive_cross_section(dem_data, wt_data, river_data, interactive=True)
